# Supplemental Frequency-Response Figures

Generates Figs. S4-S8. This notebook extends the frequency-response analysis across several STD operating points and checks that the qualitative phase and gain structure persists for finite-amplitude modulation.


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy import signal

from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURE_DIR = ROOT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Shared plotting style for manuscript PDFs.
config = {
    "font.family": "sans-serif",
    "font.size": 18.0,
    "axes.titlelocation": "left",
    "axes.titlesize": 19.0,
    "axes.labelsize": 19.0,
    "xtick.labelsize": 17.0,
    "ytick.labelsize": 17.0,
    "legend.fontsize": 17.0,
    "figure.titlesize": 19.0,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "lines.markersize": 4.0,
    "patch.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "axes.xmargin": 0.01,
    "axes.ymargin": 0.05,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 2.0,
    "ytick.minor.size": 2.0,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "legend.frameon": False,
    "legend.fancybox": False,
    "image.interpolation": "none",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}
plt.rcParams.update(config)


In [ ]:
"""
STP (Short-Term Plasticity) Sensitivity Dynamics Analysis
=========================================================

"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from dataclasses import dataclass, field
from typing import Tuple, Optional
import warnings

# -----------------------------
# Parameter container
# -----------------------------
@dataclass
class STPParams:
    """Parameters for the STP sensitivity dynamics."""
    tau_d: float = 0.5        # depression time constant [s]
    U: float = 0.15            # effective utilization parameter
    nu0: float = 10.0         # baseline presynaptic rate [Hz]
    delta_nu: float = 0.1     # modulation amplitude [Hz]

    # Derived quantities
    kappa: float = field(init=False)
    f_w0_star: float = field(init=False)
    f_U_star: float = field(init=False)

    def __post_init__(self):
        self.update_derived()

    def update_derived(self):
        """Recompute derived parameters."""
        self.kappa = 1.0 / self.tau_d + self.nu0 * self.U
        self.f_w0_star = 1.0 / (1.0 + self.tau_d * self.nu0 * self.U)
        self.f_U_star = 1.0 / (1.0 + self.tau_d * self.nu0 * self.U) ** 2

    def update(self, **kwargs):
        """Update parameters and derived quantities."""
        for key, val in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, val)
        self.update_derived()
        return self

    def copy(self, **kwargs):
        """Return a copy with selected parameter changes."""
        new_params = STPParams(
            tau_d=self.tau_d,
            U=self.U,
            nu0=self.nu0,
            delta_nu=self.delta_nu
        )
        new_params.update(**kwargs)
        return new_params

    def summary(self) -> str:
        """Return a compact parameter summary."""
        r = self.kappa * self.tau_d
        return (f"τ_d={self.tau_d:.3f}, U={self.U:.3f}, "
                f"ν₀={self.nu0:.1f}, δν={self.delta_nu:.3f}, "
                f"κ={self.kappa:.2f}, r=κτ_d={r:.2f}")


# -----------------------------
# Analytic transfer functions
# -----------------------------
def H_w0(omega: np.ndarray, p: STPParams) -> np.ndarray:
    """Transfer function for w0 sensitivity."""
    return -p.U * p.f_w0_star / (p.kappa + 1j * omega)


def H_U(omega: np.ndarray, p: STPParams) -> np.ndarray:
    """Transfer function for U sensitivity."""
    return (
        -p.U * (p.f_U_star + p.f_w0_star) / (p.kappa + 1j * omega)
        + p.U**2 * p.nu0 * p.f_w0_star / (p.kappa + 1j * omega)**2
    )


def H_C_pre_w0(omega: np.ndarray, p: STPParams) -> np.ndarray:
    """H_{C_w0}(ω) - presynaptic contribution (w0)"""
    return p.f_w0_star + H_w0(omega, p) * p.nu0


def H_C_pre_U(omega: np.ndarray, p: STPParams) -> np.ndarray:
    """H_{C_U}(ω) - presynaptic contribution (U)"""
    return p.f_U_star + H_U(omega, p) * p.nu0


# -----------------------------
# ODE simulation
# -----------------------------
def stp_sensitivity_ode(t, y, p: STPParams, omega: float):
    """ODE for STP sensitivity dynamics."""
    f_w0, f_U = y
    nu_t = p.nu0 + p.delta_nu * np.cos(omega * t)

    df_w0 = 1.0 / p.tau_d - (1.0 / p.tau_d + nu_t * p.U) * f_w0
    df_U = 1.0 / p.tau_d - (1.0 / p.tau_d + nu_t * p.U) * f_U - nu_t * p.U * f_w0

    return [df_w0, df_U]


def estimate_gain_phase(omega: float, p: STPParams,
                        n_cycles: int = 100,
                        points_per_cycle: int = 200
                        ) -> Tuple[complex, complex, complex, complex]:
    """

    Returns:
        (H_w0_sim, H_U_sim, H_C_pre_w0_sim, H_C_pre_U_sim)
    """
    T = 2.0 * np.pi / omega
    t_end = n_cycles * T
    t_eval = np.linspace(0.0, t_end, int(n_cycles * points_per_cycle))

    y0 = [p.f_w0_star, p.f_U_star]

    sol = solve_ivp(
        stp_sensitivity_ode,
        t_span=(0.0, t_end),
        y0=y0,
        t_eval=t_eval,
        args=(p, omega),
        rtol=1e-7,
        atol=1e-9,
    )

    t = sol.t
    f_w0 = sol.y[0]
    f_U = sol.y[1]

    nu_t = p.nu0 + p.delta_nu * np.cos(omega * t)
    delta_nu_t = nu_t - np.mean(nu_t)

    C_pre_w0 = f_w0 * nu_t
    C_pre_U = f_U * nu_t

    delta_f_w0 = f_w0 - np.mean(f_w0)
    delta_f_U = f_U - np.mean(f_U)
    delta_C_pre_w0 = C_pre_w0 - np.mean(C_pre_w0)
    delta_C_pre_U = C_pre_U - np.mean(C_pre_U)

    half = len(t) // 2
    t = t[half:]
    delta_nu_t = delta_nu_t[half:]
    delta_f_w0 = delta_f_w0[half:]
    delta_f_U = delta_f_U[half:]
    delta_C_pre_w0 = delta_C_pre_w0[half:]
    delta_C_pre_U = delta_C_pre_U[half:]

    dt = t[1] - t[0]
    exp_factor = np.exp(-1j * omega * t)

    F_in = np.sum(delta_nu_t * exp_factor) * dt
    F_w0 = np.sum(delta_f_w0 * exp_factor) * dt
    F_U = np.sum(delta_f_U * exp_factor) * dt
    F_C_pre_w0 = np.sum(delta_C_pre_w0 * exp_factor) * dt
    F_C_pre_U = np.sum(delta_C_pre_U * exp_factor) * dt

    return F_w0 / F_in, F_U / F_in, F_C_pre_w0 / F_in, F_C_pre_U / F_in


# -----------------------------
# Frequency-response calculation
# -----------------------------
def compute_frequency_response(p: STPParams,
                               omega_range: Tuple[float, float] = (1e-2, 1e2),
                               n_theory: int = 40,
                               n_sim: int = 15,
                               run_simulation: bool = True):
    """

    Args:

    Returns:
    """
    omega_theory = np.logspace(
        np.log10(omega_range[0]),
        np.log10(omega_range[1]),
        n_theory
    ) * p.kappa

    # Analytic transfer functions.
    Hw0 = H_w0(omega_theory, p)
    HU = H_U(omega_theory, p)
    HC_w0 = H_C_pre_w0(omega_theory, p)
    HC_U = H_C_pre_U(omega_theory, p)

    result = {
        'params': p,
        'omega_theory': omega_theory,
        'gain_w0_theory': np.abs(Hw0),
        'gain_U_theory': np.abs(HU),
        'gain_C_w0_theory': np.abs(HC_w0),
        'gain_C_U_theory': np.abs(HC_U),
        'phase_w0_theory': np.angle(Hw0),
        'phase_U_theory': np.angle(HU),
        'phase_C_w0_theory': np.angle(HC_w0),
        'phase_C_U_theory': np.angle(HC_U),
    }

    if run_simulation:
        omega_sim = np.logspace(
            np.log10(omega_range[0]),
            np.log10(omega_range[1]),
            n_sim
        ) * p.kappa

        gain_w0_sim, gain_U_sim = [], []
        phase_w0_sim, phase_U_sim = [], []
        gain_C_w0_sim, gain_C_U_sim = [], []
        phase_C_w0_sim, phase_C_U_sim = [], []

        for w in omega_sim:
            H_w0_s, H_U_s, H_C_w0_s, H_C_U_s = estimate_gain_phase(w, p)
            gain_w0_sim.append(np.abs(H_w0_s))
            gain_U_sim.append(np.abs(H_U_s))
            phase_w0_sim.append(np.angle(H_w0_s))
            phase_U_sim.append(np.angle(H_U_s))
            gain_C_w0_sim.append(np.abs(H_C_w0_s))
            gain_C_U_sim.append(np.abs(H_C_U_s))
            phase_C_w0_sim.append(np.angle(H_C_w0_s))
            phase_C_U_sim.append(np.angle(H_C_U_s))

        result.update({
            'omega_sim': omega_sim,
            'gain_w0_sim': np.array(gain_w0_sim),
            'gain_U_sim': np.array(gain_U_sim),
            'gain_C_w0_sim': np.array(gain_C_w0_sim),
            'gain_C_U_sim': np.array(gain_C_U_sim),
            'phase_w0_sim': np.array(phase_w0_sim),
            'phase_U_sim': np.array(phase_U_sim),
            'phase_C_w0_sim': np.array(phase_C_w0_sim),
            'phase_C_U_sim': np.array(phase_C_U_sim),
        })

    return result


# -----------------------------
# Plotting functions
# -----------------------------
def plot_frequency_response(freq_data: dict,
                            plot_sim: bool = True,
                            plot_sensitivity: bool = True,
                            plot_contribution: bool = True,
                            figsize: Tuple[float, float] = (10, 6)):
    """

    Args:
    """
    p = freq_data['params']
    omega_th = freq_data['omega_theory']

    n_cols = int(plot_sensitivity) + int(plot_contribution)
    if n_cols == 0:
        warnings.warn("No plots selected (both plot_sensitivity and plot_contribution are False).")
        return None, None

    fig, axes = plt.subplots(2, n_cols, figsize=figsize, sharex=True)
    if n_cols == 1:
        axes = axes.reshape(-1, 1)

    omega_sim = freq_data.get('omega_sim') if plot_sim else None
    line_w0 = None
    line_U = None
    col = 0

    if plot_sensitivity:
        ax_gain_f = axes[0, col]
        ax_phase_f = axes[1, col]

        line_w0, = ax_gain_f.loglog(
            omega_th, freq_data['gain_w0_theory'],
            label=r'$Z=w_0$', lw=2, color='C0'
        )
        line_U, = ax_gain_f.loglog(
            omega_th, freq_data['gain_U_theory'],
            label=r'$Z=U$', lw=2, color='C1'
        )

        ax_phase_f.semilogx(omega_th, freq_data['phase_w0_theory'], lw=2, color='C0')
        ax_phase_f.semilogx(omega_th, freq_data['phase_U_theory'], lw=2, color='C1')

        if omega_sim is not None:
            ax_gain_f.loglog(omega_sim, freq_data['gain_w0_sim'], 'o', ms=4, color='C0')
            ax_gain_f.loglog(omega_sim, freq_data['gain_U_sim'], 's', ms=4, color='C1')
            ax_phase_f.semilogx(omega_sim, freq_data['phase_w0_sim'], 'o', ms=4, color='C0')
            ax_phase_f.semilogx(omega_sim, freq_data['phase_U_sim'], 's', ms=4, color='C1')

        ax_gain_f.set_ylabel(r'Gain $|H_Z^f(\omega)|$')
        ax_gain_f.grid(True, which='both', ls=':')

        ax_phase_f.set_xlabel(r'Frequency $\omega$')
        ax_phase_f.set_ylabel(r'Phase $\phi_Z^f(\omega)$ [rad]')
        ax_phase_f.grid(True, which='both', ls=':')
        col += 1

    if plot_contribution:
        ax_gain_c = axes[0, col]
        ax_phase_c = axes[1, col]

        line_w0_c, = ax_gain_c.loglog(
            omega_th, freq_data['gain_C_w0_theory'],
            label=r'$Z=w_0$', lw=2, color='C0'
        )
        line_U_c, = ax_gain_c.loglog(
            omega_th, freq_data['gain_C_U_theory'],
            label=r'$Z=U$', lw=2, color='C1'
        )

        ax_phase_c.semilogx(omega_th, freq_data['phase_C_w0_theory'], lw=2, color='C0')
        ax_phase_c.semilogx(omega_th, freq_data['phase_C_U_theory'], lw=2, color='C1')

        if omega_sim is not None:
            ax_gain_c.loglog(omega_sim, freq_data['gain_C_w0_sim'], 'o', ms=4, color='C0')
            ax_gain_c.loglog(omega_sim, freq_data['gain_C_U_sim'], 's', ms=4, color='C1')
            ax_phase_c.semilogx(omega_sim, freq_data['phase_C_w0_sim'], 'o', ms=4, color='C0')
            ax_phase_c.semilogx(omega_sim, freq_data['phase_C_U_sim'], 's', ms=4, color='C1')

        if line_w0 is None:
            line_w0 = line_w0_c
            line_U = line_U_c

        ax_gain_c.set_ylabel(r'Gain $|H_Z^C(\omega)|$')
        ax_gain_c.grid(True, which='both', ls=':')

        ax_phase_c.set_xlabel(r'Frequency $\omega$')
        ax_phase_c.set_ylabel(r'Phase $\phi_Z^C(\omega)$ [rad]')
        ax_phase_c.grid(True, which='both', ls=':')

    for ax in axes.flatten():
        ax.vlines(
            p.kappa * np.array([0.1, 1.0, 10.0]),
            ymin=0, ymax=1,
            transform=ax.get_xaxis_transform(),
            color='gray', ls='--', lw=1
        )

    if line_w0 is not None and line_U is not None:
        fig.legend(
            [line_w0, line_U],
            [r'$Z=w_0$', r'$Z=U$'],
            loc='center left',
            bbox_to_anchor=(1.01, 0.5)
        )
        plt.tight_layout(rect=[0, 0, 0.97, 1])
    else:
        plt.tight_layout(rect=[0, 0, 1, 1])

    return fig, axes


# -----------------------------
# Analysis wrapper
# -----------------------------
def analyze_stp(tau_d: float = 0.5,
                U: float = 0.15,
                nu0: float = 10.0,
                delta_nu: float = 0.1,
                run_simulation: bool = True,
                show_plots: bool = True):
    """

    Args:

    Returns:
        dict with 'params', 'freq_data', 'figures'
    """
    p = STPParams(tau_d=tau_d, U=U, nu0=nu0, delta_nu=delta_nu)
    print(f"Parameters: {p.summary()}")

    freq_data = compute_frequency_response(p, run_simulation=run_simulation)

    figures = {}
    if show_plots:
        fig1, _ = plot_frequency_response(freq_data, plot_sim=run_simulation)
        figures['frequency'] = fig1
        plt.show()

    return {
        'params': p,
        'freq_data': freq_data,
        'figures': figures
    }


# -----------------------------
# Parameter comparison
# -----------------------------
def compare_parameters(param_sets: list,
                       labels: list = None,
                       figsize: Tuple[float, float] = (14, 12)):
    """

    Args:
    """
    if labels is None:
        labels = [f'Set {i+1}' for i in range(len(param_sets))]

    fig, axes = plt.subplots(4, 2, figsize=figsize, sharex=True)
    colors = plt.cm.tab10(np.linspace(0, 1, len(param_sets)))

    for i, ps in enumerate(param_sets):
        if isinstance(ps, dict):
            p = STPParams(**ps)
        else:
            p = ps

        freq_data = compute_frequency_response(p, run_simulation=False)
        omega = freq_data['omega_theory']

        axes[0, 0].loglog(
            omega, freq_data['gain_w0_theory'],
            lw=2, color=colors[i], label=labels[i]
        )
        axes[1, 0].semilogx(omega, freq_data['phase_w0_theory'], lw=2, color=colors[i])
        axes[0, 1].loglog(omega, freq_data['gain_U_theory'], lw=2, color=colors[i])
        axes[1, 1].semilogx(omega, freq_data['phase_U_theory'], lw=2, color=colors[i])

        axes[2, 0].loglog(omega, freq_data['gain_C_w0_theory'], lw=2, color=colors[i])
        axes[3, 0].semilogx(omega, freq_data['phase_C_w0_theory'], lw=2, color=colors[i])
        axes[2, 1].loglog(omega, freq_data['gain_C_U_theory'], lw=2, color=colors[i])
        axes[3, 1].semilogx(omega, freq_data['phase_C_U_theory'], lw=2, color=colors[i])

    axes[0, 0].set_ylabel(r'Gain $|H_Z^f(\omega)|$')
    axes[0, 0].set_title(r'$Z=w_0$')
    axes[0, 0].grid(True, which='both', ls=':')

    axes[1, 0].set_xlabel(r'$\omega$')
    axes[1, 0].set_ylabel(r'Phase $\phi_Z^f(\omega)$ [rad]')
    axes[1, 0].grid(True, which='both', ls=':')

    axes[0, 1].set_title(r'$Z=U$')
    axes[0, 1].grid(True, which='both', ls=':')

    axes[1, 1].set_xlabel(r'$\omega$')
    axes[1, 1].grid(True, which='both', ls=':')

    axes[2, 0].set_ylabel(r'Gain $|H_Z^C(\omega)|$')
    axes[2, 0].set_title(r'$Z=w_0$')
    axes[2, 0].grid(True, which='both', ls=':')

    axes[3, 0].set_xlabel(r'$\omega$')
    axes[3, 0].set_ylabel(r'Phase $\phi_Z^C(\omega)$ [rad]')
    axes[3, 0].grid(True, which='both', ls=':')

    axes[2, 1].set_title(r'$Z=U$')
    axes[2, 1].grid(True, which='both', ls=':')

    axes[3, 1].set_xlabel(r'$\omega$')
    axes[3, 1].grid(True, which='both', ls=':')

    handles, legend_labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc='center left', bbox_to_anchor=(1.01, 0.5))
    plt.tight_layout(rect=[0, 0, 0.97, 1])
    return fig, axes


# -----------------------------


In [ ]:
res = analyze_stp(tau_d=0.5, U=0.15, nu0=10.0, delta_nu=0.1)
fig = res['figures']['frequency']
fig.savefig(FIGURE_DIR / "STP-phase-small-r.pdf", bbox_inches='tight')


In [ ]:
res = analyze_stp(tau_d=0.5, U=0.3, nu0=10.0, delta_nu=0.1)
fig = res['figures']['frequency']
fig.savefig(FIGURE_DIR / "STP-phase-medium-r.pdf", bbox_inches='tight')


In [ ]:
res = analyze_stp(tau_d=0.5, U=1.0, nu0=10.0, delta_nu=0.1)
fig = res['figures']['frequency']
fig.savefig(FIGURE_DIR / "STP-phase-large-r.pdf", bbox_inches='tight')


In [ ]:
param_sets = [
    {'tau_d': 1.0, 'U': 1.0, 'nu0': 0.5},
    {'tau_d': 1.0, 'U': 1.0, 'nu0': 1.5},
    {'tau_d': 1.0, 'U': 1.0, 'nu0': 4.0},
]
labels = ['r=1.5', 'r=2.5', 'r=5.0']

fig, axes = compare_parameters(param_sets, labels)
plt.savefig(FIGURE_DIR / "STP-phase-multi-r.pdf", bbox_inches='tight')
plt.show()


In [ ]:
res = analyze_stp(tau_d=0.5, U=0.15, nu0=10.0, delta_nu=10.0)
fig = res['figures']['frequency']
fig.savefig(FIGURE_DIR / "STP-phase-large-delta-nu.pdf", bbox_inches='tight')
